# Backpropagation - advanced walkthrough

Run every cell from top to bottom. The notebook prints intermediate values and draws visualizations so the math stays visible.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(7)
plt.style.use('default')

## 1. Build a tiny neural network
We use XOR because a linear model cannot solve it. The point is to see the forward pass, backward pass, and gradient check.

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)

W1 = np.array([[1.0, -1.0], [1.0, -1.0]])
b1 = np.array([[0.0, 0.0]])
W2 = np.array([[1.0], [1.0]])
b2 = np.array([[0.0]])

print('X shape:', X.shape)
print('y shape:', y.shape)
print('W1 shape:', W1.shape)
print('W2 shape:', W2.shape)
display(pd.DataFrame(X, columns=['x1', 'x2']).assign(y=y.ravel()))

## 2. Forward pass with every intermediate value

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    a1 = np.tanh(z1)
    z2 = a1 @ W2 + b2
    p = sigmoid(z2)
    return z1, a1, z2, p

z1, a1, z2, p = forward(X, W1, b1, W2, b2)
print('z1 = X @ W1 + b1')
print(z1)
print('a1 = tanh(z1)')
print(a1)
print('z2 = a1 @ W2 + b2')
print(z2)
print('p = sigmoid(z2)')
print(p)

## 3. Backward pass one operation at a time

In [ ]:
m = len(X)
dz2 = (p - y) / m
dW2 = a1.T @ dz2
db2 = dz2.sum(axis=0, keepdims=True)
da1 = dz2 @ W2.T
dz1 = da1 * (1 - a1 ** 2)
dW1 = X.T @ dz1
db1 = dz1.sum(axis=0, keepdims=True)

print('dz2 shape:', dz2.shape)
print('dW2:')
print(dW2)
print('db2:', db2)
print('dW1:')
print(dW1)
print('db1:', db1)

## 4. Gradient check
A numeric gradient should match the backprop gradient for the same parameter.

In [ ]:
def loss_value(W1, b1, W2, b2):
    _, _, _, p = forward(X, W1, b1, W2, b2)
    return -np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9))

eps = 1e-5
W2_plus = W2.copy()
W2_minus = W2.copy()
W2_plus[0, 0] += eps
W2_minus[0, 0] -= eps
numeric = (loss_value(W1, b1, W2_plus, b2) - loss_value(W1, b1, W2_minus, b2)) / (2 * eps)
analytic = dW2[0, 0]
print('numeric gradient:', numeric)
print('backprop gradient:', analytic)
print('absolute difference:', abs(numeric - analytic))

## 5. Train and visualize learning

In [ ]:
W1 = np.random.normal(scale=0.8, size=(2, 3))
b1 = np.zeros((1, 3))
W2 = np.random.normal(scale=0.8, size=(3, 1))
b2 = np.zeros((1, 1))
learning_rate = 1.0
losses = []

for step in range(2500):
    z1, a1, z2, p = forward(X, W1, b1, W2, b2)
    loss = -np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9))
    losses.append(loss)
    dz2 = (p - y) / len(X)
    dW2 = a1.T @ dz2
    db2 = dz2.sum(axis=0, keepdims=True)
    da1 = dz2 @ W2.T
    dz1 = da1 * (1 - a1 ** 2)
    dW1 = X.T @ dz1
    db1 = dz1.sum(axis=0, keepdims=True)
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

_, _, _, final_p = forward(X, W1, b1, W2, b2)
print('Final probabilities:')
display(pd.DataFrame({'x1': X[:, 0], 'x2': X[:, 1], 'y': y.ravel(), 'p_hat': final_p.ravel()}))

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(losses)
ax.set_title('Backpropagation reduces XOR loss')
ax.set_xlabel('training step')
ax.set_ylabel('binary cross-entropy')
plt.show()

Try changing the hidden width from 3 to 2 or the learning rate from 1.0 to 0.2 and rerun.